# UNSW-NB15 Linear Transformer Float HLS Codegen and CSIM

This notebook converts the completed `LinearUNSWAnomalyDetector` checkpoint into an
equivalent float C++ implementation and validates 16 samples with Vitis HLS C simulation.

This stage does not run synthesis, Vivado, bitstream generation, quantization, or PYNQ.

## 2. Import dependencies

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)

Python: 3.12.7
PyTorch: 2.9.0+cpu


## 3. Path configuration

In [2]:
PROJECT_ROOT = Path("/home/cym/prj2/finn/notebooks/icl_thesis-master")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.generate_unsw_linear_hls import (
    CHECKPOINT_PATH,
    HLS_PROJECT_DIR,
    HLS_RESULTS_DIR,
    HLS_SRC_DIR,
    INPUT_DIM,
    MODEL_CONFIG_PATH,
    NUM_CLASSES,
    NUM_TEST_SAMPLES,
    ONNX_DIR,
    ONNX_PATH,
    PREPROCESS_INFO_PATH,
    SAMPLE_INPUTS_PATH,
    SAMPLE_LABELS_PATH,
    SEQ_LEN,
    export_weights,
    extract_state_dict,
    generate_project_files,
    generated_project_files,
    load_json,
    load_model_and_state,
    load_test_data,
    parse_csim_metrics,
    run_csim,
    torch_load_compatible,
    write_hls_header,
    write_hls_source,
    write_reports,
    write_tcl_scripts,
    write_test_vectors,
    write_testbench,
)

HLS_SRC_DIR.mkdir(parents=True, exist_ok=True)
HLS_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print("Checkpoint:", CHECKPOINT_PATH)
print("ONNX reference:", ONNX_PATH)
print("HLS project:", HLS_PROJECT_DIR)
print("HLS results:", HLS_RESULTS_DIR)

Checkpoint: /home/cym/prj2/finn/notebooks/icl_thesis-master/results/linear_unsw_baseline/best_model.pt
ONNX reference: /home/cym/prj2/finn/notebooks/icl_thesis-master/results/linear_unsw_baseline/onnx/unsw_linear_transformer.onnx
HLS project: /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls
HLS results: /home/cym/prj2/finn/notebooks/icl_thesis-master/results/linear_unsw_baseline/hls


## 4. Read preprocessing metadata

In [3]:
preprocess_info = load_json(PREPROCESS_INFO_PATH)
assert (preprocess_info["seq_len"], preprocess_info["input_dim"]) == (SEQ_LEN, INPUT_DIM)

print("Encoded feature dimension:", preprocess_info["one_hot_feature_dim"])
print("Sequence length:", preprocess_info["seq_len"])
print("Token input dimension:", preprocess_info["input_dim"])
print("Model sample shape:", [SEQ_LEN, INPUT_DIM])

Encoded feature dimension: 192
Sequence length: 8
Token input dimension: 24
Model sample shape: [8, 24]


## 5. Read model configuration

In [4]:
saved_model_config = load_json(MODEL_CONFIG_PATH)
print(json.dumps(saved_model_config, indent=2))

assert saved_model_config["seq_len"] == SEQ_LEN
assert saved_model_config["input_dim"] == INPUT_DIM
assert saved_model_config["num_classes"] == NUM_CLASSES

{
  "input_dim": 24,
  "seq_len": 8,
  "d_model": 16,
  "dim_feedforward": 32,
  "num_layers": 1,
  "dropout": 0.1,
  "num_classes": 2
}


## 6. Load `best_model.pt`

In [5]:
model, state_dict, model_config, checkpoint_format = load_model_and_state()

print("Model class:", model.__class__.__name__)
print("Checkpoint format:", checkpoint_format)
print("State tensors:", len(state_dict))
print("Total parameters:", sum(tensor.numel() for tensor in state_dict.values()))
print("Training mode:", model.training)

Model class: LinearUNSWAnomalyDetector
Checkpoint format: model_state_dict
State tensors: 20
Total parameters: 2770
Training mode: False


## 7. Read saved model inputs

In [6]:
test_inputs, test_labels, reference_logits, expected_predictions = load_test_data(model)

print("HLS test inputs shape:", test_inputs.shape)
print("HLS test input dtype:", test_inputs.dtype)
print("Reference logits shape:", reference_logits.shape)
print("Ground-truth labels:", test_labels.tolist())
print("Expected predictions:", expected_predictions.tolist())

HLS test inputs shape: (16, 8, 24)
HLS test input dtype: float32
Reference logits shape: (16, 2)
Ground-truth labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Expected predictions: [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1]


## 8. Read ONNX/PyTorch reference logits

In [7]:
saved_pytorch_logits_path = ONNX_DIR / "pytorch_first100_logits.npy"
saved_onnx_logits_path = ONNX_DIR / "onnx_first100_logits.npy"
saved_pytorch_logits = np.load(saved_pytorch_logits_path)[:NUM_TEST_SAMPLES]
saved_onnx_logits = np.load(saved_onnx_logits_path)[:NUM_TEST_SAMPLES]

checkpoint_vs_saved_error = float(np.max(np.abs(reference_logits - saved_pytorch_logits)))
onnx_vs_saved_error = float(np.max(np.abs(saved_onnx_logits - saved_pytorch_logits)))
print("Fresh checkpoint vs saved PyTorch max error:", checkpoint_vs_saved_error)
print("Saved ONNX vs PyTorch max error:", onnx_vs_saved_error)
print("First reference logits:", reference_logits[0])

Fresh checkpoint vs saved PyTorch max error: 2.384185791015625e-07
Saved ONNX vs PyTorch max error: 4.76837158203125e-07
First reference logits: [-1.3556116  1.2704846]


## 9. Analyze `LinearUNSWAnomalyDetector`

In [8]:
state_table = pd.DataFrame(
    [
        {
            "state_dict_key": key,
            "shape": list(tensor.shape),
            "parameters": int(tensor.numel()),
        }
        for key, tensor in state_dict.items()
    ]
)
display(state_table)
print(model)
print(
    "Forward order: input projection + position embedding -> ELU+1 Q/K -> "
    "K^T V linear attention -> residual + LayerNorm -> FFN -> residual + "
    "LayerNorm -> sequence mean -> output LayerNorm -> classifier"
)

,state_dict_key,shape,parameters
0,position_embedding,"[1, 8, 16]",128
1,input_projection.weight,"[16, 24]",384
2,input_projection.bias,[16],16
3,layers.0.attention.query.weight,"[16, 16]",256
4,layers.0.attention.key.weight,"[16, 16]",256
5,layers.0.attention.value.weight,"[16, 16]",256
6,layers.0.attention.output.weight,"[16, 16]",256
7,layers.0.attention.output.bias,[16],16
8,layers.0.feedforward.0.weight,"[32, 16]",512
9,layers.0.feedforward.0.bias,[32],32


LinearUNSWAnomalyDetector(
  (input_projection): Linear(in_features=24, out_features=16, bias=True)
  (layers): ModuleList(
    (0): _UNSWLinearEncoderBlock(
      (attention): _UNSWLinearAttention(
        (query): Linear(in_features=16, out_features=16, bias=False)
        (key): Linear(in_features=16, out_features=16, bias=False)
        (value): Linear(in_features=16, out_features=16, bias=False)
        (output): Linear(in_features=16, out_features=16, bias=True)
      )
      (feedforward): Sequential(
        (0): Linear(in_features=16, out_features=32, bias=True)
        (1): ReLU()
        (2): Dropout(p=0.1, inplace=False)
        (3): Linear(in_features=32, out_features=16, bias=True)
      )
      (norm1): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (output_norm): LayerNorm((16,), eps=1e-05, elementwise_affine=True)
  (classifier): Linear(i

## 10. Export checkpoint weights to `weights.h`

In [9]:
weight_report = export_weights(state_dict)
print("Weight tensor count:", weight_report["tensor_count"])
print("Total parameter count:", weight_report["total_parameter_count"])
print("Weights header:", weight_report["weights_header"])

Weight tensor count: 20
Total parameter count: 2770
Weights header: /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/src/weights.h


## 11. Generate the float HLS inference source

In [10]:
write_hls_header()
write_hls_source()

print((HLS_SRC_DIR / "unsw_linear_transformer.h").read_text())
print("C++ source:", HLS_SRC_DIR / "unsw_linear_transformer.cpp")
print("Top function: unsw_linear_transformer")
print("Input interface: float input[8][24]")
print("Output interface: float logits[2]")

#ifndef UNSW_LINEAR_TRANSFORMER_H
#define UNSW_LINEAR_TRANSFORMER_H

#define UNSW_SEQ_LEN 8
#define UNSW_INPUT_DIM 24
#define UNSW_D_MODEL 16
#define UNSW_FF_DIM 32
#define UNSW_NUM_CLASSES 2

void unsw_linear_transformer(
    float input[UNSW_SEQ_LEN][UNSW_INPUT_DIM],
    float logits[UNSW_NUM_CLASSES]
);

#endif

C++ source: /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/src/unsw_linear_transformer.cpp
Top function: unsw_linear_transformer
Input interface: float input[8][24]
Output interface: float logits[2]


## 12. Generate 16-sample test vectors and testbench

In [11]:
write_test_vectors(test_inputs, test_labels, reference_logits, expected_predictions)
write_testbench()

print("Test vectors:", HLS_SRC_DIR / "test_vectors.h")
print("Testbench:", HLS_SRC_DIR / "testbench.cpp")
print("Samples tested:", NUM_TEST_SAMPLES)

Test vectors: /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/src/test_vectors.h
Testbench: /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/src/testbench.cpp
Samples tested: 16


## 13. Generate Vitis HLS TCL scripts

In [12]:
write_tcl_scripts()
print("run_csim.tcl:\n")
print((HLS_PROJECT_DIR / "run_csim.tcl").read_text())
print("run_csynth.tcl generated but not executed:", HLS_PROJECT_DIR / "run_csynth.tcl")

run_csim.tcl:

open_project -reset unsw_linear_transformer_prj
set_top unsw_linear_transformer
add_files src/unsw_linear_transformer.cpp
add_files src/unsw_linear_transformer.h
add_files src/weights.h
add_files -tb src/testbench.cpp
add_files -tb src/test_vectors.h
open_solution "solution1"
set_part xc7z020clg400-1
create_clock -period 10 -name default
csim_design
exit

run_csynth.tcl generated but not executed: /home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/run_csynth.tcl


## 14. Run or verify Vitis HLS C simulation

In [13]:
csim_report = run_csim(enabled=True)
print(json.dumps(csim_report, indent=2))

if csim_report["csim_executed"] and not csim_report["csim_passed"]:
    raise RuntimeError("Vitis HLS C simulation did not pass")

{
  "vitis_hls_available": true,
  "vitis_hls_command": "/tools/Xilinx/Vitis_HLS/2022.2/bin/vitis_hls",
  "csim_executed": true,
  "csim_passed": true,
  "return_code": 0,
  "metrics": {
    "max_abs_error": 1.907348633e-06,
    "mean_abs_error": 5.280598998e-07,
    "prediction_match_rate": 1.0
  },
  "reason": "CSIM PASS (current Vitis HLS log verified)",
  "csim_log": "/home/cym/prj2/finn/notebooks/icl_thesis-master/vitis_hls/unsw_linear_transformer_hls/unsw_linear_transformer_prj/solution1/csim/report/unsw_linear_transformer_csim.log"
}


## 15. Parse C simulation output

In [14]:
csim_output_path = HLS_RESULTS_DIR / "hls_csim_output.txt"
csim_output = csim_output_path.read_text(encoding="utf-8")
parsed_metrics = parse_csim_metrics(csim_output)

print("Parsed CSIM metrics:", parsed_metrics)
print("Contains CSIM PASS:", "CSIM PASS" in csim_output)
print("Contains CSim done with 0 errors:", "CSim done with 0 errors" in csim_output)

if csim_report["csim_passed"]:
    assert parsed_metrics is not None
    assert parsed_metrics["max_abs_error"] <= 1e-3
    assert parsed_metrics["prediction_match_rate"] == 1.0

Parsed CSIM metrics: {'max_abs_error': 1.907348633e-06, 'mean_abs_error': 5.280598998e-07, 'prediction_match_rate': 1.0}
Contains CSIM PASS: True
Contains CSim done with 0 errors: True


## 16. Save HLS reports

In [15]:
write_reports(weight_report, csim_report, checkpoint_format)

required_results = [
    HLS_RESULTS_DIR / "hls_codegen_summary.md",
    HLS_RESULTS_DIR / "hls_weight_export_report.json",
    HLS_RESULTS_DIR / "hls_csim_report.json",
    HLS_RESULTS_DIR / "hls_csim_output.txt",
    HLS_RESULTS_DIR / "hls_reference_logits.npy",
    HLS_RESULTS_DIR / "hls_test_inputs.npy",
    HLS_RESULTS_DIR / "generated_file_list.txt",
]
required_project_files = generated_project_files()
for path in required_results + required_project_files:
    if not path.is_file():
        raise FileNotFoundError(f"Missing expected output: {path}")

print("CSIM passed:", csim_report["csim_passed"])
print("Can enter csynth:", csim_report["csim_passed"])
print("Generated project files:")
for path in required_project_files:
    print(" -", path.relative_to(PROJECT_ROOT))
print("Result files:")
for path in required_results:
    print(" -", path.relative_to(PROJECT_ROOT))

CSIM passed: True
Can enter csynth: True
Generated project files:
 - vitis_hls/unsw_linear_transformer_hls/src/unsw_linear_transformer.h
 - vitis_hls/unsw_linear_transformer_hls/src/unsw_linear_transformer.cpp
 - vitis_hls/unsw_linear_transformer_hls/src/weights.h
 - vitis_hls/unsw_linear_transformer_hls/src/test_vectors.h
 - vitis_hls/unsw_linear_transformer_hls/src/testbench.cpp
 - vitis_hls/unsw_linear_transformer_hls/run_csim.tcl
 - vitis_hls/unsw_linear_transformer_hls/run_csynth.tcl
 - vitis_hls/unsw_linear_transformer_hls/README.md
Result files:
 - results/linear_unsw_baseline/hls/hls_codegen_summary.md
 - results/linear_unsw_baseline/hls/hls_weight_export_report.json
 - results/linear_unsw_baseline/hls/hls_csim_report.json
 - results/linear_unsw_baseline/hls/hls_csim_output.txt
 - results/linear_unsw_baseline/hls/hls_reference_logits.npy
 - results/linear_unsw_baseline/hls/hls_test_inputs.npy
 - results/linear_unsw_baseline/hls/generated_file_list.txt
